# Cochin Hebrew manuscripts: PDF to OSIS

Converts the three Project Truth Ministries Cochin editions — Revelation and
James from Cambridge MS Oo.1.16.2 / Oo.1.32, and Matthew from Oo.1.32.

Thin driver over `pdf2osis.cochin`, covered by `tests/test_cochin.py` and
`tests/test_integration.py`.

## Why one extractor per file

The three share a house style but not a format:

| file | verses | verse header | distinguishing feature |
|---|---|---|---|
| Revelation | 406 | `Revelation N:V (Cochin N:V)` | interlinear gloss table |
| James | 107 | `James N:V (KJV N:V)` | no interlinear table |
| Matthew | 646 | `Chapter C:V` | Syriac Aramaic column |

Matthew's Syriac is set at the same type size as its English, so script rather
than size separates them. Matthew covers 1:1–19:30; the volume stops there.

Revelation is **406** records, not the 404 the previous extractor produced:
Rev 2:26 and Rev 20:12 are in the source but their headers are not set at the
usual size, so a size-keyed search missed them.

In [ ]:
from pathlib import Path

from pdf2osis import convert_pdf, get_profile

ROOT = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()
SOURCE = ROOT / "data" / "00_source_files"
OUTPUT = ROOT / "data" / "01_osis"

reports = {}
for key in ("rev", "jas", "mat"):
    profile = get_profile(key)
    reports[key] = convert_pdf(profile.default_path(SOURCE), profile, OUTPUT)
    report = reports[key]
    print(f"{key}: {report.verses} verses across {report.chapters} chapters, "
          f"{report.note_definitions} footnotes")

## Source anomalies

Verses the edition states the manuscript lacks are recorded as empty rather
than dropped, and footnotes defined but never referenced are reported.

In [ ]:
for key, report in reports.items():
    if report.empty_verses:
        print(f"{key}: {len(report.empty_verses)} verses absent from the manuscript: "
              f"{', '.join(report.empty_verses)}")
    for anomaly in report.anomalies[:5]:
        print(f"{key}: {anomaly}")

## Interlinear repair

The transcription and the gloss table are set in different subsets of one face,
and each resolves letters the other confuses — he read as het, bet as kaf.
Where a gloss repeats a transcription word at the same length and differs in
exactly one letter, the gloss wins.

In [ ]:
document = None
from pdf2osis.cochin import extract_cochin

document = extract_cochin(get_profile("rev").default_path(SOURCE), get_profile("rev"))
repaired = [r for r in document.records if r.extraction_disagreements]
print(f"{len(repaired)} verses had at least one letter corrected")
for record in repaired[:3]:
    print(f"  Rev.{record.chapter}.{record.verse}: {record.extraction_disagreements[0]}")
    print(f"    {record.hebrew[:70]}")

## The two recovered verses

Both are unambiguously in the source.

In [ ]:
by_id = {f"Rev.{r.chapter}.{r.verse}": r for r in document.records}
for verse_id in ("Rev.2.26", "Rev.20.12"):
    record = by_id[verse_id]
    print(f"{verse_id} (PDF page {record.page})")
    print(f"   {record.hebrew}")
    print(f"   {record.english[:110]}")